# Stacks, Queues, and Related Structures

## 1. Stacks

A stack is a linear data structure that follows the Last-In, First-Out (LIFO) principle. Elements are inserted and removed from the same end, called the top.

- **Push:** add element to top
- **Pop:** remove and return top element
- **Peek:** view top element without removing
- Only the top element is accessible at any time

### Python List as Stack

| Stack Operation | Python List | Time |
|----------------|-------------|------|
| `push(e)` | `L.append(e)` | O(1) amortized |
| `pop()` | `L.pop()` | O(1) amortized |
| `peek()` | `L[-1]` | O(1) |
| `isEmpty()` | `len(L) == 0` | O(1) |
| `len(S)` | `len(L)` | O(1) |

### Array-Based Stack

In [ ]:
class Stack:
    def __init__(self):
        self.data = []

    def isEmpty(self):
        return len(self.data) == 0

    def push(self, val):
        self.data.append(val)

    def pop(self):
        if self.isEmpty():
            return None
        return self.data.pop()

    def peek(self):
        if self.isEmpty():
            return None
        return self.data[-1]

    def size(self):
        return len(self.data)

#### Steps

| Step | Operation | Stack (bottom -> top) | Result |
|------|-----------|----------------------|--------|
| 1 | `S.push(5)` | `[5]` | 5 added to top |
| 2 | `S.push(3)` | `[5, 3]` | 3 added to top |
| 3 | `S.push(7)` | `[5, 3, 7]` | 7 added to top |
| 4 | `S.peek()` | `[5, 3, 7]` | returns 7, stack unchanged |
| 5 | `S.pop()` | `[5, 3]` | returns 7, removed from top |
| 6 | `S.pop()` | `[5]` | returns 3 |
| 7 | `S.isEmpty()` | `[5]` | returns False |
| 8 | `S.pop()` | `[]` | returns 5 |
| 9 | `S.isEmpty()` | `[]` | returns True |

### Linked-List Stack

In [ ]:
class Node:
    def __init__(self, val=0, nxt=None):
        self.val = val
        self.next = nxt


class LinkedStack:
    def __init__(self):
        self.top = None
        self.count = 0

    def isEmpty(self):
        return self.top is None

    def push(self, val):
        self.top = Node(val, self.top)
        self.count += 1

    def pop(self):
        if self.isEmpty():
            return None
        val = self.top.val
        self.top = self.top.next
        self.count -= 1
        return val

    def peek(self):
        if self.isEmpty():
            return None
        return self.top.val

    def size(self):
        return self.count

- **Space:** how memory is allocated and whether extra overhead exists per element.
- **Cache:** whether elements are stored contiguously, affecting CPU cache hit rate during traversal.
- **Overflow:** whether the structure can run out of capacity.

| Property | Array Stack | Linked Stack |
|----------|-------------|--------------|
| Push/Pop/Peek | O(1) | O(1) |
| Space | Fixed-size array, resized by doubling when full | Each node stores value + pointer (extra memory per element) |
| Cache | Elements contiguous in memory, fewer cache misses | Nodes scattered in memory, more cache misses |
| Overflow | Fixed capacity can overflow if not resized | No overflow, allocates nodes until system memory exhausted |

### Monotonic Stack

A monotonic stack keeps its elements sorted (increasing or decreasing) from top to bottom. Before pushing a new element, pop any elements that break the order.

- **Monotonically increasing** (top to bottom): finds the next greater element
- **Monotonically decreasing** (top to bottom): finds the next smaller element

| Property | Detail |
|---|---|
| **Key idea** | Maintain sorted order in stack to find nearest greater/smaller element |
| **Use-case** | Next greater element, daily temperatures: `[73, 74, 75, 71, 69, 72, 76]` |
| **Problem keywords** | "next greater element", "daily temperatures", "stock span" |
| **Time** | O(n) -- each element pushed and popped at most once |
| **Space** | O(n) |

#### Monotonically Increasing Stack (finds next greater to the right)

In [ ]:
def nextGreaterElement(nums):
    n = len(nums)
    res = [-1] * n
    stack = []
    for i in range(n):
        while stack and nums[i] > nums[stack[-1]]:
            idx = stack.pop()
            res[idx] = nums[i]
        stack.append(i)
    return res

#### Steps

`nums = [2, 7, 5, 4, 6, 3]`, find next greater element to the right.

| Step | Element | Operation | Stack (indices) | `res` update |
|------|---------|-----------|-----------------|-------------|
| 1 | 2 | `S.push(0)` | `[0]` | -- |
| 2 | 7 | `S.pop()` -> 0, `S.push(1)` | `[1]` | `res[0] = 7` |
| 3 | 5 | `S.push(2)` | `[1, 2]` | -- |
| 4 | 4 | `S.push(3)` | `[1, 2, 3]` | -- |
| 5 | 6 | `S.pop()` -> 3, `S.pop()` -> 2, `S.push(4)` | `[1, 4]` | `res[3] = 6`, `res[2] = 6` |
| 6 | 3 | `S.push(5)` | `[1, 4, 5]` | -- |

Remaining in stack have no next greater: `res = [7, -1, 6, 6, -1, -1]`

#### Monotonic Stack Use-Case Summary

| Goal | Stack type | When to record |
|------|-----------|----------------|
| Next greater to right | Increasing (top to bottom) | On pop (current element is the answer) |
| Next smaller to right | Decreasing (top to bottom) | On pop |
| Next greater to left | Increasing | On push (top of stack is the answer) |
| Next smaller to left | Decreasing | On push |

### Example: Bracket Matching

Push left brackets onto the stack. On a right bracket, check if the top matches. If the stack is empty at the end, all brackets are matched.

In [ ]:
def isValid(s):
    stack = []
    pairs = {")": "(", "]": "[", "}": "{"}
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in ")]}":
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
    return len(stack) == 0

### Example: Expression Evaluation

Use a stack to handle operator precedence. Push numbers for `+`/`-`, compute immediately for `*`/`/`. Sum the stack at the end.

In [ ]:
def calculate(s):
    stack = []
    num = 0
    op = "+"
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        if ch in "+-*/" or i == len(s) - 1:
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            elif op == "/":
                stack.append(int(stack.pop() / num))
            op = ch
            num = 0
    return sum(stack)

## 2. Queues

A queue is a linear data structure that follows the First-In, First-Out (FIFO) principle. Elements are added at the rear and removed from the front.

- **Enqueue:** add element to the rear
- **Dequeue:** remove and return front element
- **Front/First:** view front element without removing

### Operations

| Operation | Description | Time |
|-----------|-------------|------|
| `enqueue(e)` | Add to rear | O(1) |
| `dequeue()` | Remove from front | O(1) |
| `first()` | View front element | O(1) |
| `isEmpty()` | Check if empty | O(1) |
| `len(Q)` | Number of elements | O(1) |

### Circular Array Queue

In [ ]:
class Queue:
    def __init__(self, capacity=100):
        self.capacity = capacity + 1
        self.data = [None] * self.capacity
        self.front = 0
        self.rear = 0

    def isEmpty(self):
        return self.front == self.rear

    def isFull(self):
        return (self.rear + 1) % self.capacity == self.front

    def enqueue(self, val):
        if self.isFull():
            return
        self.rear = (self.rear + 1) % self.capacity
        self.data[self.rear] = val

    def dequeue(self):
        if self.isEmpty():
            return None
        self.front = (self.front + 1) % self.capacity
        return self.data[self.front]

    def first(self):
        if self.isEmpty():
            return None
        return self.data[(self.front + 1) % self.capacity]

#### Steps

| Step | Operation | Queue (front -> rear) | Result |
|------|-----------|----------------------|--------|
| 1 | `Q.enqueue(1)` | `[1]` | 1 added to rear |
| 2 | `Q.enqueue(2)` | `[1, 2]` | 2 added to rear |
| 3 | `Q.enqueue(3)` | `[1, 2, 3]` | 3 added to rear |
| 4 | `Q.dequeue()` | `[2, 3]` | returns 1 (FIFO) |
| 5 | `Q.dequeue()` | `[3]` | returns 2 |
| 6 | `Q.first()` | `[3]` | returns 3, queue unchanged |

**Circular:** rear wraps around via `(rear + 1) % capacity` to reuse freed slots at the front.

### Linked Queue

In [ ]:
class LinkedQueue:
    def __init__(self):
        head = Node(0)
        self.front = head
        self.rear = head

    def isEmpty(self):
        return self.front == self.rear

    def enqueue(self, val):
        node = Node(val)
        self.rear.next = node
        self.rear = node

    def dequeue(self):
        if self.isEmpty():
            return None
        node = self.front.next
        self.front.next = node.next
        if self.rear == node:
            self.rear = self.front
        return node.val

## 3. Deques (Double-Ended Queues)

A deque allows insertion and deletion at both ends. It combines the properties of stacks (LIFO at either end) and queues (FIFO).

### Operations (collections.deque)

| Operation | Description | Time |
|-----------|-------------|------|
| `D.appendleft(e)` | Add to front | O(1) |
| `D.append(e)` | Add to rear | O(1) |
| `D.popleft()` | Remove from front | O(1) |
| `D.pop()` | Remove from rear | O(1) |
| `D[0]` | Access front | O(1) |
| `D[-1]` | Access rear | O(1) |
| `D[j]` | Access by index | O(n) |
| `len(D)` | Number of elements | O(1) |
| `D.rotate(k)` | Circular shift right by k | O(k) |
| `D.remove(e)` | Remove first match | O(n) |
| `D.count(e)` | Count matches | O(n) |

### Double-Ended Queue

In [ ]:
class Node:
    def __init__(self, val=0):
        self.val = val
        self.prev = None
        self.next = None


class Deque:
    def __init__(self):
        self.head = Node()
        self.tail = Node()
        self.head.next = self.tail
        self.tail.prev = self.head
        self.count = 0

    def isEmpty(self):
        return self.count == 0

    def pushFront(self, val):
        node = Node(val)
        node.next = self.head.next
        node.prev = self.head
        self.head.next.prev = node
        self.head.next = node
        self.count += 1

    def pushBack(self, val):
        node = Node(val)
        node.prev = self.tail.prev
        node.next = self.tail
        self.tail.prev.next = node
        self.tail.prev = node
        self.count += 1

    def popFront(self):
        if self.isEmpty():
            return None
        node = self.head.next
        self.head.next = node.next
        node.next.prev = self.head
        self.count -= 1
        return node.val

    def popBack(self):
        if self.isEmpty():
            return None
        node = self.tail.prev
        self.tail.prev = node.prev
        node.prev.next = self.tail
        self.count -= 1
        return node.val

#### Steps

| Step | Operation | Deque (front -> rear) | Result |
|------|-----------|----------------------|--------|
| 1 | `D.pushFront(1)` | `[1]` | 1 at front |
| 2 | `D.pushBack(2)` | `[1, 2]` | 2 at rear |
| 3 | `D.pushFront(3)` | `[3, 1, 2]` | 3 at front |
| 4 | `D.pushBack(4)` | `[3, 1, 2, 4]` | 4 at rear |
| 5 | `D.popFront()` | `[1, 2, 4]` | returns 3 |
| 6 | `D.popBack()` | `[1, 2]` | returns 4 |

### Without Imports: List as Deque

A plain list can serve as a deque when `collections.deque` is unavailable:

| Deque operation | List equivalent | Time |
|----------------|----------------|------|
| Append to back | `dq.append(x)` | O(1) |
| Pop from back | `dq.pop()` | O(1) |
| Pop from front | `dq.pop(0)` | O(n) |
| Peek front | `dq[0]` | O(1) |
| Peek back | `dq[-1]` | O(1) |

`pop(0)` is O(n) due to shifting, but is acceptable when the total number of pops across the algorithm is bounded (e.g. each element popped at most once).

```python
dq = []
dq.append(val)      # push back
dq.pop()             # pop back
dq.pop(0)            # pop front
dq[0]                # peek front
dq[-1]               # peek back
```

## 4. Priority Queues

A priority queue dequeues the element with the highest (or lowest) priority first, regardless of insertion order.

- Regular queue: FIFO (insertion order)
- Priority queue: highest priority first

### Comparison

| Implementation | Enqueue | Dequeue (highest priority) |
|---------------|---------|---------------------------|
| Binary heap | O(log n) | O(log n) |
| Unsorted array | O(1) | O(n) |
| Sorted linked list | O(n) | O(1) |

Binary heaps are a standard implementation. Python's `heapq` module provides a min-heap.

### Binary Heap (Max-Heap)

### Pseudocode

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{siftUp} \\
\hline
\textbf{Input: } \text{heap array, index } i \text{ of newly added element} \\
\textbf{Output: } \text{element bubbled up to its correct position} \\
\\
\textbf{function } \text{siftUp}(i): \\
\quad 1: \quad \textbf{while } i > 0: \\
\quad 2: \quad\quad parent \leftarrow (i - 1) \;/\; 2 \\
\quad 3: \quad\quad \textbf{if } heap[i] > heap[parent]: \text{ swap and continue up} \\
\quad 4: \quad\quad \textbf{else: break} \\
\hline
\end{array}
$$

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{siftDown} \\
\hline
\textbf{Input: } \text{heap array, index } i \text{ of element to sink} \\
\textbf{Output: } \text{element pushed down to its correct position} \\
\\
\textbf{function } \text{siftDown}(i): \\
\quad 1: \quad \textbf{while } \text{node } i \text{ has children}: \\
\quad 2: \quad\quad \text{pick the larger child node} \\
\quad 3: \quad\quad \textbf{if } heap[i] < heap[larger]: \text{ swap and continue down} \\
\quad 4: \quad\quad \textbf{else: break} \\
\hline
\end{array}
$$

In [ ]:
class MaxHeap:
    def __init__(self):
        self.heap = []

    def push(self, val):
        self.heap.append(val)
        self._siftUp(len(self.heap) - 1)

    def pop(self):
        if not self.heap:
            return None
        self.heap[0], self.heap[-1] = self.heap[-1], self.heap[0]
        val = self.heap.pop()
        if self.heap:
            self._siftDown(0)
        return val

    def peek(self):
        return self.heap[0] if self.heap else None

    def _siftUp(self, i):
        while i > 0:
            parent = (i - 1) // 2
            if self.heap[i] > self.heap[parent]:
                self.heap[i], self.heap[parent] = self.heap[parent], self.heap[i]
                i = parent
            else:
                break

    def _siftDown(self, i):
        n = len(self.heap)
        while 2 * i + 1 < n:
            left = 2 * i + 1
            right = 2 * i + 2
            largest = i
            if left < n and self.heap[left] > self.heap[largest]:
                largest = left
            if right < n and self.heap[right] > self.heap[largest]:
                largest = right
            if largest != i:
                self.heap[i], self.heap[largest] = self.heap[largest], self.heap[i]
                i = largest
            else:
                break

#### Steps

| Step | Operation | Heap (array) | Notes |
|------|-----------|-------------|-------|
| 1 | `H.push(3)` | `[3]` | single element |
| 2 | `H.push(5)` | `[5, 3]` | 5 sifted up past 3 |
| 3 | `H.push(8)` | `[8, 3, 5]` | 8 sifted up to root |
| 4 | `H.push(1)` | `[8, 3, 5, 1]` | 1 stays at bottom |
| 5 | `H.peek()` | `[8, 3, 5, 1]` | returns 8 (max) |
| 6 | `H.pop()` | `[5, 3, 1]` | swap 8 with 1, remove 8, sift down |
| 7 | `H.pop()` | `[3, 1]` | returns 5 |

Array layout: `parent = (i-1)//2`, `left = 2i+1`, `right = 2i+2`

### Applications

- Dijkstra's shortest path
- Huffman coding
- Top-k elements
- Task scheduling by priority
- Event-driven simulation

## 5. Reference

### Structure Comparison

| Structure | Order | Insert | Remove | Access | Space |
|-----------|-------|--------|--------|--------|-------|
| Stack | LIFO | O(1) push | O(1) pop | O(1) top | O(n) |
| Queue | FIFO | O(1) enqueue | O(1) dequeue | O(1) front | O(n) |
| Deque | Both ends | O(1) either end | O(1) either end | O(1) ends | O(n) |
| Priority Queue | By priority | O(log n) | O(log n) | O(1) peek | O(n) |

### When to Use What

| Scenario | Structure |
|----------|-----------|
| Undo/redo, bracket matching, DFS | Stack |
| BFS, task scheduling, FIFO processing | Queue |
| Sliding window maximum, palindrome check | Deque |
| Top-k, shortest path, scheduling by priority | Priority Queue |
| Next greater/smaller element | Monotonic Stack |

### Pattern Quick Reference

| Pattern | Technique | Time | Use-case | Problem keywords |
|---------|-----------|------|----------|-----------------|
| Bracket matching | Push open brackets, pop on close and check match | O(n) | Validate nested delimiters | "valid parentheses", "balanced brackets" |
| Next greater element | Monotonically increasing stack, record answer on pop | O(n) | Find first larger value to the right for each element | "next greater element", "daily temperatures", "stock span" |
| Next smaller element | Monotonically decreasing stack, record answer on pop | O(n) | Find first smaller value to the right for each element | "next smaller element", "buildings with ocean view" |
| Expression evaluation | Number stack + operator precedence, compute on lower-or-equal precedence | O(n) | Evaluate infix or postfix arithmetic expressions | "basic calculator", "evaluate RPN" |
| Largest rectangle | Monotonic stack of increasing heights, compute area on pop | O(n) | Max rectangular area under a histogram or in a binary matrix | "largest rectangle in histogram", "maximal rectangle" |
| Sliding window max/min | Deque holding indices, evict out-of-window and smaller/larger values | O(n) | Track extreme value as a fixed-size window slides over the array | "sliding window maximum", "max value in window of size k" |
| BFS level-order | Queue, process all nodes at current depth before next | O(n) | Shortest unweighted path, level-by-level tree/graph traversal | "shortest path", "level order", "rotting oranges" |
| Stack-based DFS | Explicit stack instead of recursion, push neighbors | O(V + E) | Iterative depth-first graph/tree traversal | "number of islands", "flood fill" |
| Min stack | Auxiliary stack tracking the running minimum alongside the main stack | O(1) per op | O(1) getMin alongside push/pop | "min stack", "max stack" |
| Car fleet | Sort by position, stack by arrival time, pop if caught | O(n log n) | Group objects that merge on a shared path | "car fleet" |

### Cross-References

- Heap fundamentals and patterns: see `heaps.ipynb`
- Hash tables for O(1) lookup: see `hash_tables.ipynb`